Reducir estados a binarias


In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn import neighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import RobustScaler

ModuleNotFoundError: No module named 'seaborn'

# TABLA FINAL

## Calculo de Y

In [ ]:
data = pd.read_csv("final_data.csv")
data.reset_index()
data.shape

In [ ]:
review_mapping_3 = {
    5: 2,
    4: 1,
    3: 1,
    2: 0,
    1: 0
}

review_mapping_2 = {
    5: 2,
    4: 0,
    3: 0,
    2: 0,
    1: 0
}
y_5 = data["media_review_score"]
y_3 = data["media_review_score"].map(review_mapping_3)
y_2 = data["media_review_score"].map(review_mapping_2)
data["media_review_score"] = data["media_review_score"].map(review_mapping_3)
y_dict = {"y_2":y_2,
          "y_3":y_3,
          "y_5":y_5
         }

y_dict

In [ ]:
data.columns

## ANALISIS

In [ ]:
# Para un modelo de tipo KNN descartaremos las fechas, ids, y dada la baja relacion entre el estado y
# la satisfaccion se descartara los estados.
data = data.drop(columns = ["fecha_ultima_review", 
                            "order_id",
                            "order_purchase_timestamp",
                            "product_id",
                            "seller_id",
                            "seller_state",
                            "customer_state",
                            "customer_unique_id",
                            "Unnamed: 0"])
data.dtypes

In [ ]:
data.duplicated().sum()

In [ ]:
data.isna().sum()

In [ ]:
data.dtypes

In [ ]:
cols_int = ["number_payments","number_items","total_diff_items", "product_description_lenght", "product_photos_qty"]
data[cols_int] = data[cols_int].astype("int64")
data.dtypes

In [ ]:
# # Hacer histplot mas legibles
# sns.countplot(data, x = "delivered_status", hue = "media_review_score")
# plt.show()
# sns.countplot(data, x = "payment_type", hue = "media_review_score")
# plt.show()
# sns.countplot(data, x = "number_items", hue = "media_review_score")
# plt.show()
# sns.countplot(data, x = "total_diff_items", hue = "media_review_score")
# plt.show()
# sns.countplot(data, x = "product_photos_qty", hue = "media_review_score")
# plt.show()
# sns.countplot(data, x = "product_category_name_english", hue = "media_review_score")
# plt.xticks(rotation=90)  # <- AHORA VERTICAL
# plt.show()

# sns.histplot(data, x = "payment_value_sum", hue = "media_review_score")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "total_price", hue = "media_review_score")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "total_freight_value", hue = "media_review_score")
# plt.xlim(0, 500)
# plt.show()
# sns.histplot(data, x = "product_description_lenght", hue = "media_review_score")
# plt.show()
# sns.histplot(data, x = "product_weight_g", hue = "media_review_score")
# plt.show()
# sns.histplot(data, x = "product_length_cm", hue = "media_review_score")
# plt.show()
# sns.histplot(data, x = "product_height_cm", hue = "media_review_score")
# plt.show()
# sns.histplot(data, x = "product_width_cm", hue = "media_review_score")
# plt.show()
# sns.histplot(data, x = "delay_time", hue = "media_review_score")
# plt.show()

In [ ]:
data.columns

## MAPEO DE VARIABLES CATEGORICAS Y ESCALADO

In [ ]:
for col in data.columns:
    print(f"{col}: diff= {data[col].nunique()},  max= {data[col].max()},  min= {data[col].min()}")

In [ ]:
# Mapeo de categorias
mapeo_categorias = {
    "Home and Decoration":0,
    "Electronics and Technology":1,
    "Fashion and Personal Care":2,
    "Leisure, Toys and Arts":3,
    "Tools and Construction":4,
    "Miscellaneous and Other Items":5
}

data["product_category_name_english"] = data["product_category_name_english"].map(mapeo_categorias)
data["product_category_name_english"].nunique()
# MinMax de todas las variables 

In [ ]:
data["payment_type"].isna().sum()

In [ ]:
data["payment_type"].unique()

In [ ]:
# Mapeo de pagos
payment_map = {
    "multiple_payments": 4,
    "credit_card": 3,
    "debit_card": 2,
    "boleto": 1,
    "voucher": 0
}
data["payment_type"] = data["payment_type"].map(payment_map)
data["payment_type"].nunique()

In [ ]:
data["payment_type"].isna().sum()

In [ ]:
data.head()

In [ ]:
# Modificamos las medidas para obtener un volumen en cm3
data["vol"] = data["product_length_cm"] * data["product_height_cm"] * data["product_width_cm"]
data = data.drop(columns = ["product_length_cm","product_height_cm","product_width_cm"]
)

In [ ]:
data["delay_time"].min()

In [ ]:
data["delay_time"].max()

In [ ]:
data.head()

In [ ]:
data.dtypes

In [ ]:
data.isna().sum()

In [ ]:
data.columns

# CREACION DE MODELO KNN

## Analisis correlacion

In [ ]:
corr = data.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f"
)
plt.show()

In [ ]:
# Ajustamos para reducir variables intentando perder pocos datos pasando a la densidad del producto
# Eliminamos columnas con alta correlacion entre ellas, domo delivered stats y payment value sum
data["dens"] = data["product_weight_g"] / data["vol"]
data = data.drop(columns = ["payment_value_sum","product_weight_g","vol"])

In [ ]:
corr = data.corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f"
)
plt.show()

## OUTLIERS

In [ ]:
data.columns

Posibles outliers
- number_items - mas de 6 despreciables
- total_diff_items - 4
- product_photos_qty - 10 o mas
- total_freight_value mayor que 100
- dens tiene valores outliner grandes
- delay time podriamos reducirlo a entre -50 y 50 sin alterar en exceso el numero de datos

Observamos que hay una gran cantidad de outliers que pertenecen a casos reales, seria recomendable clipearlos para reducir las distancias en knn?

In [ ]:
data["number_items"] = data["delay_time"].clip(upper=5)
data["total_diff_items"] = data["total_diff_items"].clip(upper=2)
data["product_photos_qty"] = data["product_photos_qty"].clip(upper=7)
data["delay_time"] = data["delay_time"].clip(lower=-50, upper=50)

p99_price = data["total_price"].quantile(0.99)
p99_freight = data["total_freight_value"].quantile(0.99)

data["total_price"] = data["total_price"].clip(upper=p99_price)
data["total_freight_value"] = data["total_freight_value"].clip(upper=p99_freight)

data["log_total_price"] = np.log1p(data["total_price"])
data["log_total_freight_value"] = np.log1p(data["total_freight_value"])
data["log_dens"] = np.log1p(data["dens"])

In [ ]:
# Hacer histplot mas legibles
sns.countplot(data, x = "payment_type", hue = "media_review_score")
plt.show()
sns.countplot(data, x = "number_items", hue = "media_review_score")
plt.show()
sns.countplot(data, x = "total_diff_items", hue = "media_review_score")
plt.show()
sns.countplot(data, x = "product_photos_qty", hue = "media_review_score")
plt.show()
sns.countplot(data, x = "product_category_name_english", hue = "media_review_score")
plt.xticks(rotation=90)  # <- AHORA VERTICAL
plt.show()

sns.histplot(data, x = "total_price", hue = "media_review_score")
plt.xlim(0, 500)
plt.show()
sns.histplot(data, x = "total_freight_value", hue = "media_review_score")
plt.xlim(0, 500)
plt.show()
sns.histplot(data, x = "product_description_lenght", hue = "media_review_score")
plt.show()
sns.histplot(data, x = "dens", hue = "media_review_score")
plt.show()
sns.histplot(data, x = "delay_time", hue = "media_review_score")
plt.show()

## Modelo

In [ ]:
# ESCALAMOS
scaler = RobustScaler()
for col in data.columns:
    data[col] = scaler.fit_transform(data[[col]])
    print(f"{col} - val_diff: {data[col].nunique()} - val max: {data[col].max()} - val min: {data[col].min()}")

In [ ]:
# definimos funcion para generar modelos
def model_generator(X_train, X_test, y_train, y_test, n_neighbors):
    modelo = neighbors.KNeighborsClassifier(
        n_neighbors = n_neighbors,
        weights="distance"
        )
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    cv_f1 = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1_macro").mean()

    return (accuracy,f1,cv_f1)

In [ ]:
def generate_data_train_test(X,y):
    return train_test_split(
        X, y,
        train_size = 0.75,
        test_size = 0.25
    )

In [ ]:
X = data.drop(columns = "media_review_score")
X_train, X_test, y_train, y_test = generate_data_train_test(X,y_3)

In [ ]:
results = {}
for k in range(1,30):
    results[k] = model_generator(X_train, X_test, y_train, y_test, k)

data_results = pd.DataFrame.from_dict(
    results,
    orient="index",
    columns=["accuracy", "f1", "cv_f1"]
)
data_results.index.name = "neightbors"
data_results.head()

In [ ]:
data_results.loc[data_results["f1"].idxmax()]

# RESULTADOS

"uniform"
accuracy    0.531599
f1          0.434684
cv_f1       0.434154
Name: 5, dtype: float64

"distance"
accuracy    0.530659
f1          0.426348
cv_f1       0.427739
Name: 5, dtype: float
"distance" + StandardScaler
accuracy    0.551125
f1          0.434482
cv_f1       0.425284
Name: 8, dtype: floa

"uniform" + StandardScaler
accuracy    0.530904
f1          0.436421
cv_f1       0.432901
Name: 5, dtype: float
"distance" + "minkowski", p=2

--------------------------------------------------------------------------------------
Seguimos variando datos sin mejorar lo ya encontrado, nos quedamos con el mejor modelo que seria 
"distance" + StandardScaler con 8 vecinos.
accuracy    0.551125 - muy bajo
f1          0.434482 - consistente pero generaliza mal
cv_f1       0.425284

El modelo es débil, pero estable. No está roto, está limitado.

Tras varios intentos y limpiando la matriz de altas correlaciones, nos encontramos con valores muy similares e insuficientes, por lo tanto no es un modelo de calidad.

accuracy    0.555946
f1          0.437575
cv_f1       0.4359



Procedemos a intentar mejorarlo, reduciendo outliers en delay_timet64
64t6464



# MEJORA MODELO

Retiraremos las columnas que tengan valores outliners, nos quedaremso solo con los que esten entregados
- Mejora de outliers
- Extraccion de keywords de reviews